In [5]:
import sys, neo4j, pandas
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
print(sys.version)
print(neo4j.__version__)

3.10.22 (main, Oct  9 2026, 15:18:58) [MSC v.1944 64 bit (AMD64)]
6.4.1


#SETUP KODE

In [ ]:

URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "PASSWORD")   # ganti sementara dengan password asli
DB = "neo4j"

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

def run(q, **params):
    with driver.session(database=DB) as s:
        return s.run(q, **params).data()

print("Terhubung")

Terhubung


### 2.1 Wrangling dan EDA 

In [ ]:
#1.Menentukan periode waktu
P1_START, P1_END = "2015-05-12", "2015-06-26"   
P2_START, P2_END = "2015-07-06", "2015-08-10"   

Deskripsi
- P1 (training): 2015-05-12 s.d. sebelum 2015-06-26
- P2 (testing): 2015-07-06 s.d. sebelum 2015-08-10

In [8]:
#2. Filter timestamp
df = pd.read_csv("data/events.csv")
df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms")   # UTC

def ambil_periode(df, start, end):
    return df[(df["datetime"] >= start) & (df["datetime"] < end)].copy()

p1 = ambil_periode(df, P1_START, P1_END)
p2 = ambil_periode(df, P2_START, P2_END)

p1.drop(columns="datetime").to_csv("data/events_p1.csv", index=False)
p2.drop(columns="datetime").to_csv("data/events_p2.csv", index=False)

print("P1:", p1.shape, "| P2:", p2.shape)
print("Visitor di P1 dan P2:", len(set(p1.visitorid) & set(p2.visitorid)))

P1: (940430, 6) | P2: (762421, 6)
Visitor di P1 dan P2: 16174


In [9]:
#3.Subgraph
df_nodes = pd.DataFrame(run("""MATCH (n) RETURN labels(n)[0] AS jenis_simpul, 
                            count(*) AS jumlah ORDER BY jumlah DESC"""))
print(df_nodes)
df_rels = pd.DataFrame(run("""MATCH (a)-[r]->(b)
                            RETURN labels(a)[0] AS dari, type(r) AS relasi, 
                            labels(b)[0] AS ke, count(*) AS jumlah
                            ORDER BY jumlah DESC
                            """))
print(df_rels)
print("Total relasi:", df_rels["jumlah"].sum())

  jenis_simpul  jumlah
0      Visitor  479153
1         Item  145405
      dari         relasi    ke  jumlah
0  Visitor         VIEWED  Item  909730
1  Visitor  ADDED_TO_CART  Item   23038
2  Visitor         BOUGHT  Item    7662
Total relasi: 940430
